# Facial Emotion Recognition with ResNet (FER2025)

End-to-end pipeline: data download, cleaning, preprocessing, ResNet34 fine-tuning, evaluation, and export for a real-time webcam demo.

- **Task:** 6-class facial emotion classification (`angry`, `disgust`, `fear`, `happy`, `sad`, `surprise`; `neutral` is excluded)
- **Input:** grayscale face image, 128x128
- **Model:** ResNet34 (ImageNet weights), 1-channel input, custom head
- **Platform:** Kaggle (Settings > Accelerator: GPU, Internet: ON). Run the cells in order.

**Contents**

| Part | Sections |
|---|---|
| A. Setup | 0 Install and GPU check · 1 Imports and settings · 2 Reproducibility |
| B. Data | 3 Download · 4 Extraction · 5 Reading · 6 Cleaning · 7 Exploratory analysis |
| C. Preprocessing | 8 Standardization · 9 Split · 10 Normalization |
| D. Data pipeline | 11 Tensors · 12 Augmentation · 13 DataLoaders |
| E. Model | 14 Architecture · 15 Training settings · 16 Training · 17 Learning curves |
| F. Evaluation | 18 Test · 19 Confusion matrix · 20 Per-class metrics |
| G. Export | 21 Export files · 22 Webcam script · 23 Zip |


# Part A - Setup


## 0. Install and GPU check

Installs `imagehash` (perceptual hashing, used for deduplication) and checks that PyTorch sees the GPU.


In [ ]:
!pip install -q imagehash
import torch
print("PyTorch :", torch.__version__)
print("GPU disponible :", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun GPU : Runtime > Change runtime type > GPU")


## 1. Imports and settings

All the main parameters are defined here: dataset repositories, folders, excluded classes, number of images per class, and the cleaning thresholds (`MIN_SIZE`, `MIN_STD`).


In [ ]:
import os, glob, tarfile, hashlib, json, shutil, time
import numpy as np
import pandas as pd
import cv2
import imagehash
import matplotlib.pyplot as plt
from PIL import Image
from huggingface_hub import snapshot_download
from sklearn.model_selection import train_test_split

REPO_IDS = ["imadhavan/FER2025", "linkyle910125/FER2025"]
RAW_DIR = "data/fer2025"
IMG_DIR = "data/fer2025_images"
MAX_PER_CLASS = 10000  # None = toutes les images ; 5000 au depart. Monte si le disque le permet
EXCLUDE_CLASSES = ["neutral"]   # classes ignorees (non extraites, non lues)
IMG_EXT = (".jpg", ".jpeg", ".png")
MIN_STD = 5
MIN_SIZE = 32

os.makedirs("data", exist_ok=True)


## 2. Reproducibility

Fixes every random seed (Python, NumPy, PyTorch, CUDA), forces deterministic cuDNN/cuBLAS operations, and seeds the DataLoader workers.


In [ ]:
import random

SEED = 7
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # requis pour des matmul GPU deterministes

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def seed_worker(worker_id):
    # graine des workers du DataLoader (augmentation / jitter)
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed(SEED)
print("Seed fixee :", SEED)


# Part B - Data


## 3. Download

Downloads the FER2025 dataset from Hugging Face (one `.tar` archive per class). The second repository is a fallback.


In [ ]:
if not glob.glob(f"{RAW_DIR}/**/*.tar", recursive=True):
    for repo in REPO_IDS:
        try:
            snapshot_download(repo_id=repo, repo_type="dataset", local_dir=RAW_DIR)
            print("Downloaded from:", repo)
            break
        except Exception as e:
            print("Failed", repo, "->", e)

tars = glob.glob(f"{RAW_DIR}/**/*.tar", recursive=True)
print("Tar files found:", [os.path.basename(t) for t in tars])


## 4. Extraction

Extracts up to `MAX_PER_CLASS` images per class, skips excluded classes, renames files sequentially, and deletes each `.tar` afterwards to save disk space. A `.done_*` marker file makes the cell safe to re-run.


In [ ]:
DELETE_TARS = True   # supprime chaque .tar apres extraction pour liberer de la place

def disk():
    total, used, free = shutil.disk_usage(".")
    return f"libre {free / 1e9:.1f} Go / total {total / 1e9:.1f} Go"

print("Disque avant extraction :", disk())
print("Tars :", {os.path.basename(t): f"{os.path.getsize(t) / 1e9:.2f} Go" for t in tars if os.path.exists(t)})

for tar_path in tars:
    cls = os.path.basename(tar_path).replace(".tar", "").lower()
    if cls in EXCLUDE_CLASSES:
        print("Classe exclue :", cls)
        continue
    out = os.path.join(IMG_DIR, cls)
    marker = os.path.join(IMG_DIR, f".done_{cls}_{MAX_PER_CLASS}")
    if os.path.exists(marker):
        continue
    if not os.path.exists(tar_path):
        print("Tar manquant pour", cls, "-> relance la cellule de telechargement")
        continue
    shutil.rmtree(out, ignore_errors=True)
    os.makedirs(out, exist_ok=True)
    n = 0
    try:
        with tarfile.open(tar_path) as t:
            for m in t:
                if not m.isfile() or not m.name.lower().endswith(IMG_EXT):
                    continue
                data = t.extractfile(m).read()
                ext = os.path.splitext(m.name)[1]
                with open(os.path.join(out, f"{n:07d}{ext}"), "wb") as f:
                    f.write(data)
                n += 1
                if MAX_PER_CLASS and n >= MAX_PER_CLASS:
                    break
    except OSError as e:
        print(f"ERREUR sur '{cls}' apres {n} images : errno={e.errno} | {e}")
        print("Disque :", disk())
        shutil.rmtree(out, ignore_errors=True)   # evite un dossier a moitie rempli
        raise
    open(marker, "w").close()
    if DELETE_TARS:
        os.remove(tar_path)
    print("Extracted:", cls, n, "|", disk())


## 5. Reading the images

Opens every image and records its size, mode, and pixel statistics (mean, std, min, max) in `data/images_info.csv`. Unreadable files are reported and skipped.


In [ ]:
rows = []
for cls in sorted(os.listdir(IMG_DIR)):
    cls_dir = os.path.join(IMG_DIR, cls)
    if not os.path.isdir(cls_dir) or cls in EXCLUDE_CLASSES:
        continue
    for f in sorted(os.listdir(cls_dir)):
        p = os.path.join(cls_dir, f)
        try:
            img = Image.open(p)
            arr = np.array(img)
            rows.append({
                "class": cls, "path": p,
                "width": img.size[0], "height": img.size[1],
                "mode": img.mode,
                "mean": arr.mean(), "std": arr.std(),
                "min": arr.min(), "max": arr.max(),
            })
        except Exception as e:
            print("Corrupted image:", p, e)

df_raw = pd.DataFrame(rows)
df_raw.to_csv("data/images_info.csv", index=False)
print("Images read:", len(df_raw))


## 6. Cleaning

Five steps, applied in this order, each logged in a report:

1. **Too small**: width or height below `MIN_SIZE`
2. **Near uniform**: pixel standard deviation below `MIN_STD`
3. **Conflicting labels**: the same perceptual hash (pHash) appears under more than one class, all copies are removed
4. **Exact duplicates**: same MD5, first copy kept
5. **Near duplicates**: same pHash, first copy kept

Conflicts are removed *before* deduplication so that an image present in two classes is dropped instead of being kept once under an arbitrary label. The result is saved in `data/clean.csv`.


In [ ]:
df = df_raw.copy()
initial = len(df)
class_before = df["class"].value_counts()
steps = []

def log_step(name, before, after):
    removed = before - after
    steps.append({
        "step": name,
        "before": before,
        "removed": removed,
        "after": after,
        "% removed (of step)": round(removed / before * 100, 2) if before else 0,
        "% removed (of initial)": round(removed / initial * 100, 2),
        "% remaining (of initial)": round(after / initial * 100, 2),
    })

n = len(df)
df = df[(df["width"] >= MIN_SIZE) & (df["height"] >= MIN_SIZE)]
log_step("1. too small", n, len(df))

n = len(df)
df = df[df["std"] >= MIN_STD]
log_step("2. near uniform", n, len(df))

def md5_of(p):
    with open(p, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

df = df.copy()
df["md5"] = df["path"].map(md5_of)
df["phash"] = df["path"].map(lambda p: str(imagehash.phash(Image.open(p).convert("L"))))

n = len(df)
nb_labels = df.groupby("phash")["class"].transform("nunique")
df = df[nb_labels == 1]
log_step("3. conflicting labels", n, len(df))

n = len(df)
df = df.drop_duplicates(subset="md5")
log_step("4. exact duplicates", n, len(df))

n = len(df)
df = df.drop_duplicates(subset="phash")
log_step("5. near duplicates", n, len(df))

df = df.reset_index(drop=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

print(f"Images at start: {initial}")
print("=== Report per step ===")
print(pd.DataFrame(steps).to_string(index=False))
print(f"Final images: {len(df)} ({len(df) / initial * 100:.2f} % of start)")

class_after = df["class"].value_counts()
by_class = pd.DataFrame({"before": class_before, "after": class_after}).fillna(0).astype(int)
by_class["removed"] = by_class["before"] - by_class["after"]
by_class["% removed"] = (by_class["removed"] / by_class["before"] * 100).round(2)
by_class["% before (of total)"] = (by_class["before"] / by_class["before"].sum() * 100).round(2)
by_class["% after (of total)"] = (by_class["after"] / by_class["after"].sum() * 100).round(2)
print("=== Per class: before / after ===")
print(by_class)

df[["path", "class"]].to_csv("data/clean.csv", index=False)
print("Saved: data/clean.csv")


## 7. Exploratory analysis

Dataset size, image dimensions and modes, pixel statistics, class balance, brightness and contrast per class, and one sample image per class.


In [ ]:
print("=== Dataset size ===")
print("Total images:", len(df))

print("=== Dimensions (top 10) ===")
print(df.groupby(["width", "height"]).size().sort_values(ascending=False).head(10))

print("=== Image mode ===")
print(df["mode"].value_counts())

print("=== Pixel statistics ===")
print(df[["mean", "std", "min", "max"]].describe().round(2))

print("=== Brightness / contrast per class ===")
print(df.groupby("class")[["mean", "std"]].mean().round(2))

pct = (df["class"].value_counts(normalize=True) * 100).round(2)
print("=== Percentage per class ===")
print(pct)

plt.figure(figsize=(8, 5))
bars = plt.bar(pct.index, pct.values, color="steelblue")
for b, v in zip(bars, pct.values):
    plt.text(b.get_x() + b.get_width() / 2, v + 0.3, f"{v}%", ha="center")
plt.title("Percentage of each class (after cleaning)")
plt.ylabel("Percentage (%)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
df["mean"].plot(kind="hist", bins=40, ax=ax[0], title="Mean brightness")
df["std"].plot(kind="hist", bins=40, ax=ax[1], title="Contrast (std)")
plt.tight_layout()
plt.show()

classes = sorted(df["class"].unique())
fig, ax = plt.subplots(1, len(classes), figsize=(2 * len(classes), 2.5))
for a, c in zip(ax, classes):
    p = df[df["class"] == c].sample(1, random_state=0)["path"].iloc[0]
    a.imshow(Image.open(p), cmap="gray")
    a.set_title(c)
    a.axis("off")
plt.show()


# Part C - Preprocessing


## 8. Standardization

Every cleaned image is converted to grayscale, resized to `IMG_SIZE` x `IMG_SIZE` with `INTER_AREA`, and enhanced with CLAHE (local contrast equalization). Images are saved as PNG in `data/std/`. No face cropping is applied: the dataset images are already faces.


In [ ]:
IMG_SIZE = 128
USE_CLAHE = True
STD_DIR = "data/std"

shutil.rmtree(STD_DIR, ignore_errors=True)

df_clean = pd.read_csv("data/clean.csv")
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

rows = []
for i, (p, cls) in enumerate(zip(df_clean["path"], df_clean["class"])):
    img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    if USE_CLAHE:
        img = clahe.apply(img)
    out_path = os.path.join(STD_DIR, cls, f"{i:07d}.png")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    cv2.imwrite(out_path, img)
    rows.append({"path": out_path, "class": cls})
    if i % 2000 == 0:
        print(i, "/", len(df_clean))

std = pd.DataFrame(rows)
std.to_csv("data/std.csv", index=False)
print("Standardized images:", len(std))

sample = df_clean.sample(8, random_state=0)
fig, ax = plt.subplots(2, 8, figsize=(16, 4.5))
for k, p in enumerate(sample["path"]):
    a = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    b = cv2.resize(a, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    if USE_CLAHE:
        b = clahe.apply(b)
    ax[0, k].imshow(a, cmap="gray"); ax[0, k].set_title(f"{a.shape[1]}x{a.shape[0]}"); ax[0, k].axis("off")
    ax[1, k].imshow(b, cmap="gray"); ax[1, k].set_title(f"{IMG_SIZE}x{IMG_SIZE}"); ax[1, k].axis("off")
plt.tight_layout()
plt.show()


## 9. Stratified split 70 / 15 / 15

The split is done *after* cleaning and deduplication so that duplicates cannot leak between train, validation and test. It is stratified by class and seeded.


In [ ]:
train_df, temp_df = train_test_split(std, test_size=0.30, stratify=std["class"], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["class"], random_state=SEED)

print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))
print(pd.DataFrame({
    "train": train_df["class"].value_counts(normalize=True) * 100,
    "val": val_df["class"].value_counts(normalize=True) * 100,
    "test": test_df["class"].value_counts(normalize=True) * 100,
}).round(2))

os.makedirs("data/splits_final", exist_ok=True)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    d[["path", "class"]].to_csv(f"data/splits_final/{name}.csv", index=False)
print("Saved: data/splits_final/")


## 10. Normalization

Pixels are scaled to [0, 1], then standardized with the mean and standard deviation of the **training set only**; the same values are applied to validation and test. Arrays are stored as float16 in `data/arrays/`, and the statistics are saved in `data/norm_stats.json` (reused by the webcam script).


In [ ]:
class_names = sorted(train_df["class"].unique())
label_map = {c: k for k, c in enumerate(class_names)}

X_train = np.stack([cv2.imread(p, cv2.IMREAD_GRAYSCALE) for p in train_df["path"]]).astype("float32") / 255.0
mean, sd = float(X_train.mean()), float(X_train.std())
print("Train mean:", round(mean, 4), "| Train std:", round(sd, 4))

os.makedirs("data/arrays", exist_ok=True)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    X = X_train if name == "train" else (
        np.stack([cv2.imread(p, cv2.IMREAD_GRAYSCALE) for p in d["path"]]).astype("float32") / 255.0)
    X = ((X - mean) / sd)[..., np.newaxis]
    y = d["class"].map(label_map).values.astype("int64")
    np.save(f"data/arrays/X_{name}.npy", X.astype("float16"))
    np.save(f"data/arrays/y_{name}.npy", y)
    print(f"{name}: X {X.shape} | mean {X.mean():.3f} | std {X.std():.3f}")

with open("data/norm_stats.json", "w") as f:
    json.dump({"img_size": IMG_SIZE, "use_clahe": USE_CLAHE,
               "mean": mean, "std": sd, "classes": class_names}, f, indent=2)
print("Saved: data/arrays/ and data/norm_stats.json")


# Part D - Data pipeline


## 11. Loading the arrays into PyTorch

Converts the arrays to float32 tensors with shape `(N, 1, 128, 128)`.


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

X_train = torch.from_numpy(np.load("data/arrays/X_train.npy").astype("float32")).permute(0, 3, 1, 2)
X_val   = torch.from_numpy(np.load("data/arrays/X_val.npy").astype("float32")).permute(0, 3, 1, 2)
X_test  = torch.from_numpy(np.load("data/arrays/X_test.npy").astype("float32")).permute(0, 3, 1, 2)
y_train = torch.from_numpy(np.load("data/arrays/y_train.npy"))
y_val   = torch.from_numpy(np.load("data/arrays/y_val.npy"))
y_test  = torch.from_numpy(np.load("data/arrays/y_test.npy"))

with open("data/norm_stats.json") as f:
    class_names = json.load(f)["classes"]

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)


## 12. Augmentation (training set only)

Random horizontal flip, random affine transform (rotation, translation, scale), random erasing, and brightness/contrast jitter. Validation and test data are never augmented.


In [ ]:
train_aug = v2.Compose([
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    v2.RandomErasing(p=0.25, scale=(0.02, 0.15)),   # masque un petit rectangle aleatoire
])

class EmotionDataset(Dataset):
    def __init__(self, X, y, aug=None, jitter=False):
        self.X, self.y, self.aug, self.jitter = X, y, aug, jitter

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        x = self.X[i]
        if self.aug is not None:
            x = self.aug(x)
        if self.jitter:
            x = x * (1 + 0.2 * (torch.rand(1) * 2 - 1)) + 0.2 * (torch.rand(1) * 2 - 1)
        return x, self.y[i]

train_ds = EmotionDataset(X_train, y_train, aug=train_aug, jitter=True)
val_ds   = EmotionDataset(X_val, y_val)
test_ds  = EmotionDataset(X_test, y_test)

idx = [0, 400, 800, 1200, 1600, 2000, 2400, 2800]
fig, ax = plt.subplots(4, len(idx), figsize=(2 * len(idx), 8))
for k, i in enumerate(idx):
    ax[0, k].imshow(X_train[i, 0], cmap="gray"); ax[0, k].axis("off")
    ax[0, k].set_title(class_names[int(y_train[i])])
    for r in range(1, 4):
        ax[r, k].imshow(train_ds[i][0][0], cmap="gray"); ax[r, k].axis("off")
plt.tight_layout()
plt.show()


## 13. DataLoaders

Batch size 128. The training loader is shuffled with a seeded generator so that the order is reproducible.


In [ ]:
BATCH_SIZE = 128
g = torch.Generator()
g.manual_seed(SEED)   # ordre du shuffle reproductible
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True,
                          worker_init_fn=seed_worker, generator=g)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
                          worker_init_fn=seed_worker)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True,
                          worker_init_fn=seed_worker)

xb, yb = next(iter(train_loader))
print("Batch:", xb.shape, yb.shape)
print("Mean:", round(xb.mean().item(), 3), "| Std:", round(xb.std().item(), 3))


# Part E - Model


## 14. ResNet architecture

ResNet34 from torchvision with four changes: a 1-channel input convolution (initialized from the pretrained RGB filters summed over channels), an optional max-pool, a deterministic global average pooling (`GlobalMean`), and a new classification head (`Dropout(0.4)` + `Linear`).


In [ ]:
import torch.nn as nn
from torchvision import models

ARCH = "resnet34"       # "resnet18" ou "resnet34"
PRETRAINED = True       # True = poids ImageNet (fine-tuning), False = depuis zero
USE_MAXPOOL = True      # True = ~4x plus rapide ; False = cartes de features plus grandes mais plus lent

class GlobalMean(nn.Module):
    # equivalent de AdaptiveAvgPool2d(1), mais deterministe sur GPU
    def forward(self, x):
        return x.mean(dim=(2, 3), keepdim=True)

def build_resnet(n_classes, arch=ARCH, pretrained=PRETRAINED):
    m = models.get_model(arch, weights="DEFAULT" if pretrained else None)

    # entree 1 canal (niveaux de gris) au lieu de 3 (RGB)
    old = m.conv1
    m.conv1 = nn.Conv2d(1, old.out_channels, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        m.conv1.weight.data = old.weight.data.sum(dim=1, keepdim=True)  # somme des canaux RGB

    # optionnel : retirer le maxpool garde des cartes de features plus grandes (plus lent)
    if not USE_MAXPOOL:
        m.maxpool = nn.Identity()
    m.avgpool = GlobalMean()

    # nouvelle tete de classification
    m.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.fc.in_features, n_classes))
    return m

set_seed(SEED)   # memes poids initiaux (tete) a chaque execution
device = "cuda" if torch.cuda.is_available() else "cpu"
model = build_resnet(len(class_names)).to(device)
print("Device:", device, "|", ARCH, "| pretrained:", PRETRAINED)
print("Parameters:", sum(p.numel() for p in model.parameters()))

model.eval()   # eval + no_grad : ne modifie pas les statistiques BatchNorm
with torch.no_grad():
    print("Output shape:", model(torch.zeros(2, 1, IMG_SIZE, IMG_SIZE).to(device)).shape)
model.train()


## 15. Training settings

AdamW with weight decay, cosine annealing learning-rate schedule, cross-entropy loss, MixUp, and mixed precision (fp16) on GPU.


In [ ]:
EPOCHS = 100
PATIENCE = 7   # un peu plus patient : MixUp rend la val loss plus bruitee
LR = 10e-4 if PRETRAINED else 1e-3   # fine-tuning : LR plus faible

criterion = nn.CrossEntropyLoss(label_smoothing=0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=5e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# MixUp : melange deux images du batch (et leurs labels) pendant l'entrainement seulement
MIXUP_ALPHA = 0.4   # force du melange (0 = desactive ; 0.2 doux, 0.4 moyen)
MIXUP_PROB  = 0.5   # proportion de batchs melanges

use_amp = (device == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)   # precision mixte (plus rapide sur GPU)

def run_epoch(loader, train):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for xb, yb in loader:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)

            mixed = train and MIXUP_ALPHA > 0 and np.random.rand() < MIXUP_PROB
            if mixed:
                lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
                perm = torch.randperm(xb.size(0), device=xb.device)
                xb = lam * xb + (1 - lam) * xb[perm]
                yb2 = yb[perm]

            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                out = model(xb)
                if mixed:
                    loss = lam * criterion(out, yb) + (1 - lam) * criterion(out, yb2)
                else:
                    loss = criterion(out, yb)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * len(yb)
            pred = out.argmax(1)
            if mixed:   # accuracy ponderee entre les deux labels du melange
                correct += (lam * (pred == yb).sum() + (1 - lam) * (pred == yb2).sum()).item()
            else:
                correct += (pred == yb).sum().item()
            n += len(yb)
    return total_loss / n, correct / n


## 16. Training

Trains with early stopping on the validation loss and saves the best checkpoint to `models/best_resnet.pt`.


In [ ]:
os.makedirs("models", exist_ok=True)
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val, wait = float("inf"), 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    scheduler.step()

    for k, v in zip(history, [tr_loss, va_loss, tr_acc, va_acc]):
        history[k].append(v)
    print(f"Epoch {epoch:02d} | train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
          f"val loss {va_loss:.3f} acc {va_acc:.3f} | {time.time() - t0:.0f}s")

    if va_loss < best_val:
        best_val, wait = va_loss, 0
        torch.save(model.state_dict(), "models/best_resnet.pt")
    else:
        wait += 1
        if wait >= PATIENCE:
            print("Early stopping.")
            break


## 17. Learning curves


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history["train_acc"], label="train"); ax[1].plot(history["val_acc"], label="val")
ax[1].set_title("Accuracy"); ax[1].legend()
plt.tight_layout()
plt.show()


# Part F - Evaluation


## 18. Test set

Loads the best checkpoint and evaluates on the test set. Optional test-time augmentation (`USE_TTA`) averages the predictions of an image and its mirror.


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

model.load_state_dict(torch.load("models/best_resnet.pt", map_location=device))
model.eval()

USE_TTA = False   # True = moyenne de l'image et de son miroir (souvent +1 a 2 points)

def predict(loader, tta=False):
    probs, ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            p = model(xb).float().softmax(1)
            if tta:
                p = (p + model(torch.flip(xb, dims=[3])).float().softmax(1)) / 2
            probs.append(p.cpu())
            ys.append(yb)
    return torch.cat(ys).numpy(), torch.cat(probs).numpy()

y_true, y_prob = predict(test_loader, tta=USE_TTA)
y_pred = y_prob.argmax(1)

print(f"Test accuracy : {accuracy_score(y_true, y_pred):.4f}  (TTA = {USE_TTA})\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))


## 19. Confusion matrix

Raw counts and row-normalized version, plus the most frequent confusions.


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)   # par ligne : part de chaque vraie classe (= recall)

fig, ax = plt.subplots(1, 2, figsize=(14, 5.5))
for a, M, title, fmt in [(ax[0], cm, "Matrice de confusion (effectifs)", "d"),
                         (ax[1], cm_norm, "Matrice de confusion (normalisee par ligne)", ".2f")]:
    im = a.imshow(M, cmap="Blues")
    a.set_xticks(range(len(class_names))); a.set_xticklabels(class_names, rotation=45, ha="right")
    a.set_yticks(range(len(class_names))); a.set_yticklabels(class_names)
    a.set_xlabel("Classe predite"); a.set_ylabel("Vraie classe"); a.set_title(title)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            a.text(j, i, format(M[i, j], fmt), ha="center", va="center",
                   color="white" if M[i, j] > M.max() / 2 else "black")
    plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout()
plt.show()

# 5 confusions les plus frequentes (hors diagonale)
pairs = [(cm[i, j], class_names[i], class_names[j])
         for i in range(len(class_names)) for j in range(len(class_names)) if i != j]
print("Confusions les plus frequentes (vrai -> predit) :")
for n, t, p in sorted(pairs, reverse=True)[:5]:
    print(f"  {t:>9} -> {p:<9} : {n}")


## 20. Per-class metrics

Precision, recall and F1 for each class.


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=range(len(class_names)))

x = np.arange(len(class_names)); w = 0.27
plt.figure(figsize=(10, 4.5))
plt.bar(x - w, prec, w, label="precision")
plt.bar(x, rec, w, label="recall")
plt.bar(x + w, f1, w, label="F1")
plt.xticks(x, class_names, rotation=30)
plt.ylim(0, 1); plt.axhline(f1.mean(), color="gray", ls="--", lw=1, label=f"F1 moyen {f1.mean():.2f}")
plt.title("Performance par classe (test)")
plt.legend(); plt.tight_layout(); plt.show()


# Part G - Export for the webcam demo


## 21. Export model and configuration

Copies the checkpoint and writes `webcam_config.json` (classes, image size, normalization statistics, architecture).


In [ ]:
os.makedirs("export", exist_ok=True)
shutil.copy("models/best_resnet.pt", "export/best_resnet.pt")

with open("data/norm_stats.json") as f:
    cfg = json.load(f)
cfg["arch"] = ARCH
cfg["use_maxpool"] = USE_MAXPOOL
cfg["src_size"] = 75   # taille native des images FER2025

with open("export/webcam_config.json", "w") as f:
    json.dump(cfg, f, indent=2)

print("Config :", cfg)
print("Fichiers dans export/ :", os.listdir("export"))


## 22. Generate `webcam.py`

Writes the real-time demo script (face detection, same preprocessing as training, ResNet prediction). It is also provided separately in the repository.


In [ ]:
webcam_code = r'''import argparse
import json
import sys

import cv2
import numpy as np
import torch
import torch.nn as nn
from torchvision import models

parser = argparse.ArgumentParser()
parser.add_argument("--camera", type=int, default=None, help="webcam index (0, 1, 2...)")
args = parser.parse_args()

cfg = json.load(open("webcam_config.json"))
CLASSES = cfg["classes"]
IMG_SIZE = cfg["img_size"]
SRC_SIZE = cfg.get("src_size", 75)   # native size of the FER2025 images (75x75)
MIN_CONF = 0.40                      # below this: "uncertain" (the model has no neutral class)
device = "cuda" if torch.cuda.is_available() else "cpu"


class GlobalMean(nn.Module):
    def forward(self, x):
        return x.mean(dim=(2, 3), keepdim=True)


def build_resnet(n_classes):
    m = models.get_model(cfg["arch"], weights=None)
    m.conv1 = nn.Conv2d(1, m.conv1.out_channels, kernel_size=7, stride=2, padding=3, bias=False)
    if not cfg["use_maxpool"]:
        m.maxpool = nn.Identity()
    m.avgpool = GlobalMean()
    m.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(m.fc.in_features, n_classes))
    return m


model = build_resnet(len(CLASSES))
model.load_state_dict(torch.load("best_resnet.pt", map_location=device))
model.to(device).eval()

# --- face detector -----------------------------------------------------------
if not hasattr(cv2, "data"):
    raise SystemExit("OpenCV without Haar cascades (headless or 5.x build?). Fix with:\n"
                     "  pip uninstall -y opencv-python opencv-python-headless\n  pip install \"opencv-python<5\"")
face_det = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")
if face_det.empty():
    raise SystemExit("Could not load haarcascade_frontalface_default.xml")
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))


def preprocess(gray_face):
    # same chain as training: native 75x75 -> 128x128 (INTER_AREA) -> CLAHE.
    # Going through 75x75 first gives the same blur as the training images.
    img = cv2.resize(gray_face, (SRC_SIZE, SRC_SIZE), interpolation=cv2.INTER_AREA)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    if cfg["use_clahe"]:
        img = clahe.apply(img)
    x = (img.astype("float32") / 255.0 - cfg["mean"]) / cfg["std"]
    return torch.from_numpy(x)[None, None].to(device)


def square_crop(gray, x0, y0, w, h, margin=0.10):
    # SQUARE crop (no distortion when resizing), kept inside the image
    H, W = gray.shape
    cx, cy = x0 + w // 2, y0 + h // 2
    half = int(max(w, h) * (0.5 + margin))
    half = min(half, cx, cy, W - cx, H - cy)
    return gray[cy - half:cy + half, cx - half:cx + half], (cx - half, cy - half, cx + half, cy + half)


# --- opening the webcam ------------------------------------------------------
def open_camera(index):
    if sys.platform.startswith("win"):
        backends = [cv2.CAP_DSHOW, cv2.CAP_MSMF, cv2.CAP_ANY]
    elif sys.platform == "darwin":
        backends = [cv2.CAP_AVFOUNDATION, cv2.CAP_ANY]
    else:
        backends = [cv2.CAP_V4L2, cv2.CAP_ANY]
    indexes = [index] if index is not None else [0, 1, 2]
    for i in indexes:
        for b in backends:
            cap = cv2.VideoCapture(i, b)
            if not cap.isOpened():
                cap.release()
                continue
            cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
            cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
            good = False
            for _ in range(30):               # the first frames are often empty/black
                ok, frame = cap.read()
                if ok and frame is not None and frame.mean() > 1:
                    good = True
                    break
            if good:
                print(f"Webcam opened: index {i}, backend {cap.getBackendName()}")
                return cap
            cap.release()
    return None


cap = open_camera(args.camera)
if cap is None:
    raise SystemExit("Could not open the webcam. Check: (1) no other app is using it "
                     "(Zoom, Teams, browser), (2) camera permission (Windows: Settings > Privacy > "
                     "Camera; macOS: Settings > Privacy > Camera for Terminal/VS Code), "
                     "(3) try  python webcam.py --camera 1")

smooth = None
fails = 0
print("Press q to quit")

while True:
    ok, frame = cap.read()
    if not ok or frame is None:
        fails += 1
        if fails > 30:
            print("The webcam stopped returning frames.")
            break
        continue
    fails = 0

    frame = cv2.flip(frame, 1)   # mirror view (text stays readable)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_det.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(80, 80))

    if len(faces) > 0:
        x0, y0, w, h = max(faces, key=lambda f: f[2] * f[3])
        face, (x1, y1, x2, y2) = square_crop(gray, x0, y0, w, h)

        if face.size > 0:
            with torch.inference_mode():
                p = model(preprocess(face)).softmax(1)[0].cpu().numpy()
            smooth = p if smooth is None else 0.6 * smooth + 0.4 * p

            k = int(smooth.argmax())
            label = CLASSES[k] if smooth[k] >= MIN_CONF else "uncertain"
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f"{label} {smooth[k]:.0%}", (x1, max(y1 - 10, 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
            for i, (c, v) in enumerate(zip(CLASSES, smooth)):
                cv2.rectangle(frame, (10, 10 + 22 * i), (10 + int(150 * v), 28 + 22 * i), (0, 200, 255), -1)
                cv2.putText(frame, c, (15, 25 + 22 * i), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    else:
        smooth = None
        cv2.putText(frame, "No face detected", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    cv2.imshow("Emotions", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
'''

with open("export/webcam.py", "w") as f:
    f.write(webcam_code)
print("Ecrit : export/webcam.py")


## 23. Zip for download

Bundles the files into `emotion_webcam.zip` in `/kaggle/working` (Output tab > Download). Then run `python webcam.py` locally.


In [ ]:
import zipfile

# Kaggle : tout ce qui est dans /kaggle/working apparait dans l'onglet Output (a droite) -> Download
os.makedirs("/kaggle/working", exist_ok=True)
zip_path = "/kaggle/working/emotion_webcam.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ["best_resnet.pt", "webcam_config.json", "webcam.py"]:
        z.write(os.path.join("export", name), arcname=name)

print("Contenu du zip :", zipfile.ZipFile(zip_path).namelist())
print("Telecharge-le : Save Version > Save & Run All, puis onglet Output > emotion_webcam.zip")
